In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{linear function} $$

$$ 
\mathbb{R}^m \times \mathbb{R}^n \to \mathbb{R}^n, \quad \mathbf{y}(\mathbf{w}, \mathbf{b}) = 
X\mathbf{w} + \mathbf{b}, \quad X \in \mathbb{R}^{n \times m}
$$

$$ \frac{\partial \mathbf{y}}{\partial \mathbf{w}} = X $$

$$ \frac{\partial \mathbf{y}}{\partial \mathbf{b}} = \mathbf{I}_{n} $$

$$ 
d\mathbf{y} = 
\frac{\partial \mathbf{y}}{\partial \mathbf{w}} d\mathbf{w} + 
\frac{\partial \mathbf{y}}{\partial \mathbf{b}} d\mathbf{b} 
$$

$$ \text{Frobenius equation} $$

$$ 
\left \langle \frac{\partial F}{\partial \mathbf{w}}, d\mathbf{w} \right \rangle _F + 
\left \langle \frac{\partial F}{\partial \mathbf{b}}, d\mathbf{b} \right \rangle _F =
\left \langle \frac{\partial F}{\partial \mathbf{y}}, d\mathbf{y} \right \rangle _F =
$$

$$ 
\left \langle \frac{\partial F}{\partial \mathbf{y}}, \frac{\partial \mathbf{y}}{\partial \mathbf{w}} d\mathbf{w} + 
\frac{\partial \mathbf{y}}{\partial \mathbf{b}} d\mathbf{b} \right \rangle _F =
$$

$$ 
\left \langle \frac{\partial F}{\partial \mathbf{y}}, \frac{\partial \mathbf{y}}{\partial \mathbf{w}} d\mathbf{w} \right \rangle _F + 
\left \langle \frac{\partial F}{\partial \mathbf{y}}, \frac{\partial \mathbf{y}}{\partial \mathbf{b}} d\mathbf{b} \right \rangle _F =
$$

$$
\left \langle \Bigg( \frac{\partial \mathbf{y}}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial F}{\partial \mathbf{y}}, d\mathbf{w} \right \rangle _F + 
\left \langle \Bigg( \frac{\partial \mathbf{y}}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial F}{\partial \mathbf{y}}, d\mathbf{b} \right \rangle _F \implies 
$$

$$ 
\begin{dcases}
    \frac{\partial F}{\partial \mathbf{w}} = \Bigg( \frac{\partial \mathbf{y}}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial F}{\partial \mathbf{y}} \\
    \\
    \frac{\partial F}{\partial \mathbf{b}} = \Bigg( \frac{\partial \mathbf{y}}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial F}{\partial \mathbf{y}}
\end{dcases}
$$

In [ ]:
import torch.autograd as autograd
import torch.nn as nn
import torch as tr

import import_ipynb
from approx import approx # type: ignore


def linear(X: tr.Tensor, w: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
    """Linear function: y = X @ w + b."""

    return tr.addmm(b, X, w)


class LinearFunction(autograd.Function):
    """Custom autograd function for the `linear`."""

    @staticmethod
    def forward(ctx, X: tr.Tensor, w: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
        ctx.save_for_backward(X)
        return linear(X, w, b)

    @staticmethod
    def backward(ctx, dF_dy: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor, tr.Tensor]:
        (X, ) = ctx.saved_tensors

        dy_dw = X
        dF_dw = dy_dw.t() @ dF_dy          
        dF_db = dF_dy.sum(dim=0)     
        
        return (None, dF_dw, dF_db)
    
class Linear(nn.Module):
    """Custom module for the affine linear transformation."""

    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `W` has shape (out_features, in_features) to be consistent with `nn.Linear`
        if W is None:
            self.weight = nn.Parameter(0.1 * tr.randn(out_features, in_features))
        else:
            self.weight = nn.Parameter(W.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(0.1 * tr.randn(out_features, ))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

    def forward(self, x):
        assert x.shape[1] == self.weight.shape[1]
        
        return LinearFunction.apply(x, self.weight.t(), self.bias)

In [ ]:
# Unit tests

def test_gradcheck(S: int, FI: int, FO: int) -> None:
    def fn(x, W, b):
        return LinearFunction.apply(x, W, b)

    x = tr.randn(S, FI, dtype=tr.float64, requires_grad=False)
    W = tr.randn(FI, FO, dtype=tr.float64, requires_grad=True)
    b = tr.randn(FO, dtype=tr.float64, requires_grad=True)

    assert autograd.gradcheck(fn, (x, W, b), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(S: int, FI: int, FO: int) -> None:
    nn_model = nn.Linear(FI, FO)
    dj_model = Linear(FI, FO, nn_model.weight, nn_model.bias)

    x = tr.randn(S, FI, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = dj_model(x1).sum()
    y1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = nn_model(x2).sum()
    y2.backward()

    assert x1 == approx(x2)
    assert dj_model.weight == approx(nn_model.weight)
    assert dj_model.weight.grad == approx(nn_model.weight.grad)
    assert dj_model.bias == approx(nn_model.bias)
    assert dj_model.bias.grad == approx(nn_model.bias.grad)


if __name__ == "__main__":
    test_gradcheck(1, 1, 1)
    test_gradcheck(10, 1, 1)
    test_gradcheck(10, 20, 1)
    test_gradcheck(10, 30, 30)

    test_compare(1, 1, 1)
    test_compare(10, 1, 1)
    test_compare(10, 20, 1)
    test_compare(10, 30, 30)
